# ТЕМА 9. Агентні системи — виправлений навчальний ноутбук

**Мета:** побудувати email-агента в LangGraph із правильною маршрутизацією та `human-in-the-loop`, а потім створити невелику RAG-систему й оцінити її за допомогою RAGAS.

> **Навчальна демонстрація, не production-рішення.** Надсилання email та створення bug ticket **імітуються**: жодних реальних листів і тікетів цей ноутбук не створює. Приклади про пам'ятки України — спрощений навчальний корпус, а не перевірена історична енциклопедія.

### Що виправлено порівняно з оригіналом

1. API-ключ береться з environment/Colab Secrets, а не записується у файл.
2. За замовчуванням увімкнено **деморежим без платних API-викликів**. Щоб використати свій OpenAI API-ключ, змініть `RUN_LIVE_API = True` у конфігурації.
3. Типи стану узгоджено з фактичними даними: `NotRequired`, `AnyMessage`, `add_messages`, `ticket_id`.
4. Для billing/critical листів **спочатку створюється чернетка, а потім запитується схвалення**.
5. `human_review` працює з `interrupt()` / `Command(resume=...)`; чекпоінтер — `InMemorySaver`.
6. `RetrievalQA` замінено явною схемою retrieval → контекст → generation, щоб кроки були зрозумілі.
7. Тестові запитання RAG мають **реальні еталонні відповіді**; додано negative case.
8. RAGAS використовує сучасний **collections API**; чотири метрики оцінюються окремо, без вигаданого єдиного score.
9. Додано docstrings, коментарі та безплатні самоперевірки.

**Як запускати:** Google Colab або Jupyter з Python 3.11+. Виконуйте комірки зверху вниз. Після перемикання режиму перезапустіть kernel/runtime та запустіть комірки з початку.

## 0. Пакети та параметри виконання

Комірка встановлення може працювати кілька хвилин. Версії обмежено *сумісними major-версіями*, але вони не є точним lock-файлом — у навчальному середовищі результати можуть дещо змінюватися після оновлень. Після встановлення на Colab за потреби перезапустіть runtime.

**Увага до витрат:** режим `RUN_LIVE_API=True` використовує платні виклики OpenAI для класифікації, генерації та embeddings. Оцінювання RAGAS робить **додаткові** LLM/embedding-запити і тому має окремий перемикач, який за замовчуванням вимкнений.

In [ ]:
%pip install -q "langgraph>=1.0,<2" "langchain-openai>=1.0,<2" "langchain-core>=1.0,<2" "langchain-community>=0.4,<1" "langchain-text-splitters>=1.0,<2" "faiss-cpu>=1.9" "ragas>=0.4,<1" "openai>=2,<3" "pandas>=2,<3"

### Конфігурація: деморежим або OpenAI API

- `RUN_LIVE_API=False`: **безплатна демонстрація**. LLM замінено контрольованими навчальними відповідями; пошук використовує прості локальні embeddings. Це дає змогу виконати весь notebook без ключа.
- `RUN_LIVE_API=True`: справжні виклики `ChatOpenAI` і `OpenAIEmbeddings` (платні).
- `RUN_RAGAS_EVAL=True`: окремо запускає **платне** оцінювання чотирма LLM-based метриками; потрібне `RUN_LIVE_API=True`.
- `MAX_EVAL_CASES` обмежує кількість прикладів у платному оцінюванні.

Ключ **не друкується**. Для Colab додайте `OPENAI_API_KEY` у **Secrets**; для Jupyter локально — встановіть environment variable. Якщо ключ відсутній у live-режимі, з'явиться приховане поле `getpass()`.

In [ ]:
import os
from getpass import getpass
from importlib.metadata import version

# === ЗМІНІТЬ ЛИШЕ ЦІ ПАРАМЕТРИ ===
RUN_LIVE_API = False       # True => реальні платні OpenAI API calls
RUN_RAGAS_EVAL = False     # True => додаткові платні виклики LLM-as-a-Judge
MAX_EVAL_CASES = 2        # скільки test cases відправляти на RAGAS оцінювання
MODEL_NAME = "gpt-4o-mini"
EVAL_MODEL_NAME = "gpt-4o-mini"
EMBEDDING_MODEL_NAME = "text-embedding-3-small"
# =================================

# Не передаємо листи/контекст у сторонній сервіс трасування без явної потреби.
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

if RUN_RAGAS_EVAL and not RUN_LIVE_API:
    raise ValueError("Для RAGAS-викликів потрібне RUN_LIVE_API=True")

if RUN_LIVE_API and not os.environ.get("OPENAI_API_KEY"):
    try:
        from google.colab import userdata
        secret = userdata.get("OPENAI_API_KEY")
        if secret:
            os.environ["OPENAI_API_KEY"] = secret
    except (ImportError, KeyError, AttributeError):
        pass

if RUN_LIVE_API and not os.environ.get("OPENAI_API_KEY"):
    secret = getpass("OPENAI_API_KEY (введення приховане): ").strip()
    if not secret:
        raise ValueError("OPENAI_API_KEY відсутній. Задайте ключ або RUN_LIVE_API=False")
    os.environ["OPENAI_API_KEY"] = secret

print("Режим:", "OpenAI API (платно)" if RUN_LIVE_API else "ДЕМО (без API-запитів)")
print("RAGAS:", "увімкнено" if RUN_RAGAS_EVAL else "вимкнено")
for package in ("langgraph", "langchain-openai", "ragas"):
    try:
        print(f"{package}: {version(package)}")
    except Exception:
        print(f"{package}: версію не знайдено — перевірте встановлення")

## Частина 1. Email-агент підтримки клієнтів (LangGraph)

### 1.1. State (стан) і вузли

- **Схема стану** описується `TypedDict`, але сам поточний стан здебільшого є `dict`.
- `NotRequired[T]` означає, що ключ може **бути відсутнім**. `T | None` означає, що значення може дорівнювати `None` — це не одне й те саме.
- `messages` містить **об'єкти повідомлень**, а не `list[str]`. Reducer `add_messages` дозволяє додавати повідомлення, а не замінювати весь список.
- **Node** — Python callable, що приймає стан та повертає **лише оновлення**; `Command` також може визначити маршрут.

`intent` — тип звернення (`billing`, `question`, `bug`, `feature`, `complex`), а не спеціальна властивість LangGraph.

In [ ]:
from typing import Annotated, Literal
from typing_extensions import NotRequired, TypedDict
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.types import Command, interrupt
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI

class EmailClassification(TypedDict):
    """Структура результату класифікації одного email.

    intent: мета звернення; urgency: критичність; topic: тема;
    summary: короткий зміст. Це схема типів, а не виклик LLM.
    """
    intent: Literal["question", "bug", "billing", "feature", "complex"]
    urgency: Literal["low", "medium", "high", "critical"]
    topic: str
    summary: str

class EmailAgentState(TypedDict):
    """Спільний стан LangGraph: вхідні поля + результати вузлів.

    Вхідні ключі обов'язкові; поля, які з'являються в процесі роботи,
    позначено NotRequired. Reducer add_messages об'єднує повідомлення.
    """
    email_content: str
    sender_email: str
    email_id: str
    messages: Annotated[list[AnyMessage], add_messages]
    classification: NotRequired[EmailClassification]
    search_results: NotRequired[list[str]]
    customer_history: NotRequired[dict]
    ticket_id: NotRequired[str]
    draft_response: NotRequired[str]
    delivery_status: NotRequired[str]

# Безплатний режим не створює LLM-клієнт і не читає ключ.
llm = (
    ChatOpenAI(model=MODEL_NAME, temperature=0, timeout=45, max_retries=1)
    if RUN_LIVE_API else None
)

### 1.2. Класифікація й маршрутизація

У деморежимі результат класифікації **запрограмовано** для кількох типових прикладів. У live-режимі використовується реальний OpenAI `with_structured_output()`.

Зверни увагу: **термінова оплата не йде відразу на `human_review`**. Спочатку `draft_response` створює чернетку. Також `bug_tracking` не пропускається для критичного bug report.

In [ ]:
def read_email(state: EmailAgentState) -> dict:
    """Зареєструвати отриманий email у списку повідомлень стану.

    Ця функція НЕ читає реальну поштову скриньку. Вхідний лист уже
    передано в state['email_content']; тут створюється HumanMessage.
    """
    return {"messages": [HumanMessage(content=state["email_content"])]}


def demo_classify_email(content: str) -> EmailClassification:
    """Детермінований mock класифікатора без OpenAI API.

    Це прості правила для навчального прикладу, НЕ повноцінне NLP.
    """
    lower = content.lower()
    if any(word in lower for word in ("оплат", "списал", "стягнул", "підписк", "refund")):
        return {"intent": "billing", "urgency": "critical", "topic": "оплата", "summary": "Проблема оплати"}
    if any(word in lower for word in ("помилк", "збій", "crash", "експорт", "не працює")):
        return {"intent": "bug", "urgency": "high", "topic": "помилка застосунку", "summary": "Повідомлення про баг"}
    if any(word in lower for word in ("додайте", "можете додати", "темний режим")):
        return {"intent": "feature", "urgency": "low", "topic": "нова функція", "summary": "Запит на нову функцію"}
    if any(word in lower for word in ("504", "api", "інтеграц")):
        return {"intent": "complex", "urgency": "high", "topic": "технічне питання", "summary": "Складний технічний запит"}
    return {"intent": "question", "urgency": "low", "topic": "користувацьке питання", "summary": "Звичайне питання"}


def classify_intent(
    state: EmailAgentState,
) -> Command[Literal["search_documentation", "draft_response", "bug_tracking"]]:
    """Класифікувати лист та обрати маршрут, не пропускаючи draft_response.

    Command(update=...) записує класифікацію у стан.
    Command(goto=...) визначає наступний вузол у workflow.
    """
    if RUN_LIVE_API:
        structured_llm = llm.with_structured_output(EmailClassification, method="json_schema")
        prompt = (
            "Класифікуй email підтримки за intent, urgency, topic, summary. "
            "Можливі intent: question, bug, billing, feature, complex. "
            "Не виконуй жодних дій; лише класифікуй.\n"
            f"Email: {state['email_content']}\nВід: {state['sender_email']}"
        )
        classification = structured_llm.invoke(prompt)
    else:
        classification = demo_classify_email(state["email_content"])

    if classification["intent"] == "bug":
        goto = "bug_tracking"
    elif classification["intent"] in ("question", "feature"):
        goto = "search_documentation"
    else:
        goto = "draft_response"
    return Command(update={"classification": classification}, goto=goto)

### 1.3. Вузли даних та дій

У цьому ноутбуці пошук документації та створення bug ticket **локальні mock-операції** в обох режимах. Підключення OpenAI API **не робить** ці дії справжніми.

У реальній системі тут з'явилися б retriever/CRM/Jira та обробка транзитних помилок. `RetryPolicy` потрібен для тимчасових збоїв зовнішніх сервісів; для локального mock він не потрібен. Повторювані дії на кшталт створення тікета треба робити ідемпотентними.

In [ ]:
# Крихітна навчальна база знань, без зовнішніх API.
DEMO_KNOWLEDGE_BASE = {
    "question": [
        "Відновлення пароля: відкрийте сторінку входу й натисніть «Забули пароль».",
        "Якщо лист не надійшов, перевірте папку Спам або зверніться до підтримки.",
    ],
    "feature": [
        "Запити на нові функції збирає продуктова команда; терміни реалізації не обіцяються.",
    ],
}


def search_documentation(state: EmailAgentState) -> Command[Literal["draft_response"]]:
    """Отримати навчальні фрагменти з локальної бази знань.

    Це mock retrieval, а не справжній векторний пошук (його буде в частині 2).
    """
    intent = state["classification"]["intent"]
    results = DEMO_KNOWLEDGE_BASE.get(intent, [])
    return Command(update={"search_results": results}, goto="draft_response")


def bug_tracking(state: EmailAgentState) -> Command[Literal["draft_response"]]:
    """Створити *імітацію* тікета і зберегти його ID у відповідному полі.

    Демо-ID не підтверджує існування тікета в Jira чи іншому сервісі.
    """
    ticket_id = f"DEMO-BUG-{state['email_id']}"
    return Command(update={"ticket_id": ticket_id}, goto="draft_response")

### 1.4. Створення чернетки та обов'язкове погодження

У live-режимі LLM отримує інструкцію **не вигадувати** ініційоване повернення коштів, виконані операції або терміни. Ця інструкція знижує ризик, але не гарантує правильності: саме тому важливий людський перегляд.

`billing`, `complex`, `high` і `critical` вимагають перегляду вже **готової** чернетки. Навіть якщо email критичний, спершу генеруємо draft, а потім викликаємо `interrupt()`.

In [ ]:
def demo_draft_reply(state: EmailAgentState) -> str:
    """Повернути безпечну навчальну чернетку без реального LLM."""
    intent = state["classification"]["intent"]
    if intent == "billing":
        return ("Вітаємо! Дякуємо за повідомлення про можливе подвійне списання. "
                "Ми передамо звернення спеціалісту для перевірки платежів. "
                "Наразі повернення коштів НЕ підтверджене й НЕ ініційоване в цьому демо.")
    if intent == "bug":
        return ("Дякуємо за повідомлення про помилку. "
                f"Створено тільки навчальний запис {state.get('ticket_id', 'DEMO')}; "
                "жодного реального тікета не створено.")
    if intent == "feature":
        return ("Дякуємо за пропозицію! Ми можемо передати її продуктовій команді, "
                "але наразі не можемо назвати дату реалізації.")
    if intent == "complex":
        return "Дякуємо за опис проблеми. Ваше питання потребує аналізу фахівця."
    return "Відкрийте сторінку входу та натисніть «Забули пароль», щоб відновити доступ."


def draft_response(
    state: EmailAgentState,
) -> Command[Literal["human_review", "send_reply"]]:
    """Скласти відповідь з контексту і вирішити, чи потрібна людина.

    Не змінює безпосередньо state: повертає Command(update=..., goto=...).
    """
    classification = state["classification"]
    docs_text = "\n".join(f"- {doc}" for doc in state.get("search_results", []))
    ticket_note = f"Демо-ID: {state['ticket_id']}" if state.get("ticket_id") else ""

    if RUN_LIVE_API:
        messages = [
            SystemMessage(content=(
                "Ти складаєш ЧЕРНЕТКУ відповіді українською для підтримки клієнтів. "
                "Враховуй лише отриманий контекст. Якщо даних бракує, прямо це скажи. "
                "Не стверджуй, що гроші повернуті, refund ініційований, email відправлений, "
                "або реальний bug ticket створений: ці дії тут НЕ виконуються. "
                "Не вигадуй факти, гарантії та терміни."
            )),
            HumanMessage(content=(
                f"Email клієнта: {state['email_content']}\n"
                f"Намір: {classification['intent']}\n"
                f"Терміновість: {classification['urgency']}\n"
                f"Документація (навчальний приклад):\n{docs_text}\n{ticket_note}"
            )),
        ]
        response_text = str(llm.invoke(messages).content)
    else:
        response_text = demo_draft_reply(state)

    needs_review = (
        classification["urgency"] in ("high", "critical")
        or classification["intent"] in ("billing", "complex")
    )
    goto = "human_review" if needs_review else "send_reply"
    return Command(update={"draft_response": response_text}, goto=goto)

### 1.5. Human-in-the-loop (`interrupt`)

Вузол призупиняється й віддає payload людині. Для відновлення використовується `Command(resume=...)` **з тим самим `thread_id`**.

**Важливо:** при відновленні LangGraph запускає вузол `human_review` з початку; тому жодних незворотних зовнішніх дій перед `interrupt()` тут немає. У відхиленому сценарії workflow завершується без надсилання.

In [ ]:
def human_review(
    state: EmailAgentState,
) -> Command[Literal["send_reply", "__end__"]]:
    """Зупинити workflow до рішення людини та обробити approval/rejection.

    interrupt() серіалізує payload; після Command(resume=...) повертає
    введений людиною dict. Вузол запускається повторно від початку.
    """
    # Намагаймося не виконувати side effects до interrupt().
    decision = interrupt({
        "email_id": state["email_id"],
        "original_email": state["email_content"],
        "draft_response": state["draft_response"],
        "intent": state["classification"]["intent"],
        "urgency": state["classification"]["urgency"],
        "action": "Перевірте й схваліть / відредагуйте чернетку або відхиліть її",
    })
    if not isinstance(decision, dict) or not isinstance(decision.get("approved"), bool):
        raise ValueError("Очікується dict з булевим полем approved")
    if not decision["approved"]:
        return Command(update={"delivery_status": "rejected_by_human"}, goto=END)

    edited = decision.get("edited_response")
    final_draft = edited if isinstance(edited, str) and edited.strip() else state["draft_response"]
    return Command(update={"draft_response": final_draft}, goto="send_reply")


def send_reply(state: EmailAgentState) -> dict:
    """Імітувати надсилання листа та позначити статус.

    НЕ викликає SMTP/Gmail API; не створює зовнішнього side effect.
    """
    draft = state.get("draft_response", "").strip()
    if not draft:
        raise ValueError("Неможливо надіслати порожню чернетку")
    print("[SIMULATION ONLY] Email НЕ відправлено. Попередній перегляд:")
    print(draft[:250])
    return {"delivery_status": "simulated_sent"}

### 1.6. Побудова графа та checkpoint

`InMemorySaver` зберігає стан **лише у пам'яті процесу**. Після перезапуску kernel він зникне. Для довготривалого зберігання існують окремі стійкі checkpointers (SQLite/Postgres тощо).

Маршрутизація: фіксовані переходи — `add_edge`, динамічні — `Command(goto=...)` всередині функцій. Не додавайте паралельно статичне ребро й `Command(goto)` для того самого переходу, якщо не хочете виконання обох маршрутів.

In [ ]:
workflow = StateGraph(EmailAgentState)
workflow.add_node("read_email", read_email)
workflow.add_node("classify_intent", classify_intent)
workflow.add_node("search_documentation", search_documentation)
workflow.add_node("bug_tracking", bug_tracking)
workflow.add_node("draft_response", draft_response)
workflow.add_node("human_review", human_review)
workflow.add_node("send_reply", send_reply)

workflow.add_edge(START, "read_email")
workflow.add_edge("read_email", "classify_intent")
workflow.add_edge("send_reply", END)

checkpointer = InMemorySaver()
app = workflow.compile(checkpointer=checkpointer)
print("Email workflow скомпільовано")

### 1.7. Запуск: критичне звернення про оплату

Очікуваний потік: `read_email → classify_intent → draft_response → human_review (pause) → send_reply`. На відміну від початкового ноутбука, `draft_response` **не порожня** в момент погодження.

In [ ]:
from uuid import uuid4

initial_state = {
    "email_content": "З мене двічі стягнули плату за підписку! Це терміново!",
    "sender_email": "customer@example.com",
    "email_id": "email_123",
    "messages": [],
}
# Новий thread_id для кожного демонстраційного запуску, щоб не змішувати checkpoints.
config = {"configurable": {"thread_id": f"topic9-billing-{uuid4().hex[:8]}"}}
paused_result = app.invoke(initial_state, config=config)

assert "__interrupt__" in paused_result, "Очікували паузу для людського перегляду"
assert paused_result.get("draft_response"), "Чернетка має бути готовою ДО human_review"
assert paused_result["classification"]["intent"] == "billing"
print("Агент призупинений для погодження. Чернетка:\n")
print(paused_result["draft_response"])
print("\nДеталі запиту на погодження:", paused_result["__interrupt__"][0].value)

### 1.8. Погодження та відновлення

Відредагована відповідь **не стверджує**, що повернення вже ініційоване. Після `resume` надсилання лише імітується.

In [ ]:
human_response = Command(resume={
    "approved": True,
    "edited_response": (
        "Дякуємо за повідомлення про можливе подвійне списання. "
        "Передаємо звернення на перевірку спеціалісту. "
        "Будь-яке повернення коштів потребує окремого підтвердження."
    ),
})
final_result = app.invoke(human_response, config=config)
assert final_result["delivery_status"] == "simulated_sent"
print("Статус:", final_result["delivery_status"], "(справжній email не надсилали)")

### 1.9. Безплатний smoke test маршрутизації (необов'язково)

У деморежимі перевіряємо, що: звичайний запит не потребує погодження; критичний bug report реєструє демо-тікет **перед** погодженням. У live-режимі цей тест пропущено, щоб не виконувати додаткові платні запити.

In [ ]:
if not RUN_LIVE_API:
    normal = app.invoke(
        {"email_content": "Як скинути пароль?", "sender_email": "a@example.com", "email_id": "normal", "messages": []},
        config={"configurable": {"thread_id": f"normal-{uuid4().hex[:8]}"}},
    )
    assert "__interrupt__" not in normal
    assert normal["classification"]["intent"] == "question"
    assert normal["delivery_status"] == "simulated_sent"

    bug = app.invoke(
        {"email_content": "Помилка: експорт PDF не працює", "sender_email": "b@example.com", "email_id": "bug1", "messages": []},
        config={"configurable": {"thread_id": f"bug-{uuid4().hex[:8]}"}},
    )
    assert "__interrupt__" in bug
    assert bug["ticket_id"] == "DEMO-BUG-bug1"
    assert bug["draft_response"]
    print("Smoke tests пройдено: question → simulated send; bug → DEMO ticket → review")
else:
    print("Smoke tests пропущено в live-режимі для економії API-викликів")

## Частина 2. RAG: Retrieval → Generation → Evaluation

RAG = **Retrieval-Augmented Generation**. Спочатку retriever знаходить фрагменти документів, потім LLM формує відповідь на основі отриманого контексту. Тут **не використовуємо застарілий `RetrievalQA`**: кроки показано окремими функціями.

### 2.1. Невеликий навчальний корпус

Висловлювання в цьому корпусі використовуються **як тестові дані**, а не як історичне джерело істини. Еталонні відповіді нижче побудовано *лише* з цього корпусу, щоб можна було перевіряти Faithfulness (відповідність контексту) і Recall (повноту retrieval).

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

texts = [
    "Софійський собор у Києві було побудовано в 1037 році за часів Ярослава Мудрого.",
    "Києво-Печерська лавра — монастир, заснований у 1051 році. У цьому уривку засновник не названий.",
    "Замок у Кам'янці-Подільському — фортеця, побудована в XIV столітті.",
    "Львівська опера — театр опери та балету, відкритий у 1900 році.",
    "Хотинська фортеця розташована на березі Дністра.",
    "Андріївська церква в Києві — пам'ятка архітектури XVIII століття.",
    "Палац Потоцьких у Львові — пам'ятка архітектури XIX століття.",
]

# ID документа потрібен для відстеження retrieval і тестів Hit@k.
docs = [
    Document(page_content=text, metadata={"doc_id": f"doc_{i+1:02d}", "source": "demo_corpus"})
    for i, text in enumerate(texts)
]
print(f"У корпусі {len(docs)} коротких документів")

### 2.2. Chunking, embeddings і FAISS

- `RecursiveCharacterTextSplitter` ділить довгі документи на фрагменти. Тут документи короткі, тому більшість залишиться одним chunk — це очікувано.
- `OpenAIEmbeddings` у live-режимі створює **семантичні** embeddings через платний API.
- У деморежимі `DemoCharNgramEmbeddings` створює локальні хешовані символьні embeddings **лише для перевірки механіки FAISS**. Це не заміна семантичній моделі, retrieval quality у цьому режимі нерепрезентативна.

In [ ]:
import hashlib
import math
import re
from langchain_core.embeddings import Embeddings

class DemoCharNgramEmbeddings(Embeddings):
    """Безплатні детерміновані символьні n-грамні вектори для smoke test.

    УВАГА: не семантична embedding-модель. Пошук тестує лише механіку
    FAISS і потік даних, а не реальну якість retrieval.
    """
    def __init__(self, dimensions: int = 256, ngram: int = 3):
        self.dimensions = dimensions
        self.ngram = ngram

    def _vectorize(self, text: str) -> list[float]:
        """Побудувати нормалізований hashing-vector із символьних n-грам."""
        normalized = re.sub(r"\s+", " ", text.lower()).strip()
        vec = [0.0] * self.dimensions
        if not normalized:
            return vec
        grams = [normalized[i:i+self.ngram] for i in range(max(1, len(normalized)-self.ngram+1))]
        for gram in grams:
            position = int.from_bytes(hashlib.blake2b(gram.encode("utf-8"), digest_size=4).digest(), "big") % self.dimensions
            vec[position] += 1.0
        norm = math.sqrt(sum(x * x for x in vec)) or 1.0
        return [value / norm for value in vec]

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        """Векторизувати всі фрагменти корпусу (без API)."""
        return [self._vectorize(text) for text in texts]

    def embed_query(self, text: str) -> list[float]:
        """Векторизувати пошуковий запит (без API)."""
        return self._vectorize(text)

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40)
chunks = splitter.split_documents(docs)
embeddings = (
    OpenAIEmbeddings(model=EMBEDDING_MODEL_NAME, max_retries=1)
    if RUN_LIVE_API else DemoCharNgramEmbeddings()
)
vectorstore = FAISS.from_documents(chunks, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print(f"FAISS: {len(chunks)} chunks, топ-3 за запитом")

### 2.3. Генерація відповіді з retrieved context

У live-режимі виклик OpenAI передає **тільки** запит і знайдений контекст та явно просить утримуватися від непідтверджених тверджень. У деморежимі `demo_rag_answer` повертає **заздалегідь запрограмовану** відповідь — це не результат LLM.

In [ ]:
rag_llm = ChatOpenAI(model=MODEL_NAME, temperature=0, max_retries=1) if RUN_LIVE_API else None


def demo_rag_answer(question: str, source_documents: list[Document]) -> str:
    """Навчальна rule-based відповідь за вмістом знайдених документів.

    Не є LLM і не використовується для оцінки реальної якості генерації.
    """
    q = question.lower()
    # Негативний сценарій: засновника корпус не називає.
    if "хто" in q and "заснув" in q:
        return "У наданих документах не названо засновника Києво-Печерської лаври."
    examples = [
        ("софій", "софій", "Згідно з навчальним корпусом, Софійський собор побудовано у 1037 році."),
        ("коли засновано києво", "києво", "Згідно з навчальним корпусом, Києво-Печерську лавру засновано у 1051 році."),
        ("хотин", "хотин", "Згідно з навчальним корпусом, Хотинська фортеця розташована на березі Дністра."),
        ("львівськ", "львівськ", "Згідно з навчальним корпусом, Львівську оперу відкрили у 1900 році."),
    ]
    combined = " ".join(doc.page_content.lower() for doc in source_documents)
    for query_marker, source_marker, answer in examples:
        if query_marker in q and source_marker in combined:
            return answer
    return "У наданому контексті недостатньо даних, щоб відповісти на це питання."


def answer_with_rag(question: str) -> dict:
    """Повернути питання, відповідь та *реально знайдені* chunks для evaluation.

    На відміну від RetrievalQA, retriever і LLM тут розділені явно.
    """
    source_documents = retriever.invoke(question)
    contexts = [doc.page_content for doc in source_documents]
    if RUN_LIVE_API:
        context_text = "\n\n".join(f"[Фрагмент {i+1}] {text}" for i, text in enumerate(contexts))
        answer = str(rag_llm.invoke([
            SystemMessage(content=(
                "Відповідай українською ТІЛЬКИ за наданими фрагментами. "
                "Якщо потрібного факту немає, прямо скажи про це. "
                "Не додавай зовнішніх знань і не вигадуй джерел."
            )),
            HumanMessage(content=f"Запитання: {question}\n\nКонтекст:\n{context_text}"),
        ]).content)
    else:
        answer = demo_rag_answer(question, source_documents)
    return {
        "question": question,
        "answer": answer,
        "contexts": contexts,
        "source_documents": source_documents,
    }

example = answer_with_rag("Коли було побудовано Софійський собор?")
print("Запитання:", example["question"])
print("Відповідь:", example["answer"])
print("Знайдені chunks (за рангом):")
for doc in example["source_documents"]:
    print("-", doc.metadata.get("doc_id"), doc.page_content)

### 2.4. Golden dataset: тестові запитання та еталони

**Виправлено критичний недолік:** `reference` (раніше `ground_truth`) тепер **не порожній**. Він потрібен для `ContextPrecision` та `ContextRecall` у reference-based версії RAGAS.

Останній приклад — **negative test**: корпус не містить імені засновника. Його запускаємо окремо, щоб перевірити здатність агента утриматися від вигадування відповіді; для основного LLM-evaluation використаємо три позитивні приклади. Таке відділення необхідне, бо стандартний reference-based Context Recall не є прямою метрикою здатності відмовитися від відповіді.

In [ ]:
# Датасет пов'язаний із конкретними фрагментами demo-корпусу.
golden_cases = [
    {
        "question": "Коли було побудовано Софійський собор?",
        "reference": "За навчальним корпусом, Софійський собор побудовано в 1037 році.",
        "expected_doc_id": "doc_01",
    },
    {
        "question": "Коли засновано Києво-Печерську лавру?",
        "reference": "Києво-Печерську лавру засновано у 1051 році.",
        "expected_doc_id": "doc_02",
    },
    {
        "question": "Де розташована Хотинська фортеця?",
        "reference": "Хотинська фортеця розташована на березі Дністра.",
        "expected_doc_id": "doc_05",
    },
]

# Однакові inputs і gold references при повторних експериментах.
test_data = []
for case in golden_cases:
    output = answer_with_rag(case["question"])
    test_data.append({
        "question": case["question"],
        "response": output["answer"],
        "retrieved_contexts": output["contexts"],
        "reference": case["reference"],
        "retrieved_doc_ids": [doc.metadata["doc_id"] for doc in output["source_documents"]],
        "expected_doc_id": case["expected_doc_id"],
    })

assert all(row["reference"].strip() for row in test_data), "Відсутній golden reference"
assert len(test_data) == len(golden_cases)

# Безплатна детермінована перевірка Hit@3 за doc_id.
# Це НЕ Contextual Recall і не якісна оцінка LLM-as-a-Judge.
hits = sum(row["expected_doc_id"] in row["retrieved_doc_ids"] for row in test_data)
print(f"Retrieval Hit@3 (demo corpus): {hits}/{len(test_data)}")
for row in test_data:
    print(f"• {row['question']}\n  answer: {row['response']}\n  reference: {row['reference']}")

negative_question = "Хто заснував Києво-Печерську лавру?"
negative_result = answer_with_rag(negative_question)
print("\nNegative test (відповідь не має вигадувати засновника):")
print(negative_result["answer"])

### 2.5. RAGAS: чотири окремі метрики (опційні платні виклики)

Використовуємо **collections API**, а не застарілі `ragas.evaluate` / `ragas.metrics.faithfulness`:

- `ContextPrecision` — чи ранжуються релевантні chunks вище нерелевантних (потрібен reference).
- `ContextRecall` — чи покриває retrieved context факти еталонної відповіді (потрібен reference).
- `Faithfulness` — чи підтримуються твердження generated response знайденим контекстом. **Не тотожна фактичній правильності джерел.**
- `AnswerRelevancy` — чи відповідає generated response на user query. **Не перевіряє істинність фактів.**

Одна метрика може використовувати **кілька внутрішніх викликів** LLM (і embeddings для Answer Relevancy), тому `MAX_EVAL_CASES=2`, а `RUN_RAGAS_EVAL=False` за замовчуванням. Порівнюйте не лише усереднення, а й **провальні окремі приклади**.

Джерела: [RAGAS Context Precision](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/context_precision/), [Context Recall](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/context_recall/), [Faithfulness](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/faithfulness/), [Answer Relevancy](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/answer_relevance/).

In [ ]:
import pandas as pd

if RUN_RAGAS_EVAL:
    from openai import AsyncOpenAI
    from ragas.llms import llm_factory
    from ragas.embeddings.base import embedding_factory
    from ragas.metrics.collections import (
        ContextPrecision, ContextRecall, Faithfulness, AnswerRelevancy
    )

    # Суддя відокремлений від генератора (навіть якщо model id однаковий).
    # max_retries=1 обмежує неочікувані повторні платні виклики.
    eval_client = AsyncOpenAI(timeout=60.0, max_retries=1)
    evaluator_llm = llm_factory(EVAL_MODEL_NAME, client=eval_client)
    evaluator_embeddings = embedding_factory(
        "openai", model=EMBEDDING_MODEL_NAME, client=eval_client
    )
    scorers = {
        "context_precision": ContextPrecision(llm=evaluator_llm),
        "context_recall": ContextRecall(llm=evaluator_llm),
        "faithfulness": Faithfulness(llm=evaluator_llm),
        "answer_relevancy": AnswerRelevancy(
            llm=evaluator_llm, embeddings=evaluator_embeddings
        ),
    }

    scores = []
    for case in test_data[:MAX_EVAL_CASES]:
        question = case["question"]
        contexts = case["retrieved_contexts"]
        reference = case["reference"]
        response = case["response"]
        # RAGAS сам робить платні виклики; 4 метрики виконуються окремо.
        cp = await scorers["context_precision"].ascore(
            user_input=question, reference=reference, retrieved_contexts=contexts
        )
        cr = await scorers["context_recall"].ascore(
            user_input=question, reference=reference, retrieved_contexts=contexts
        )
        faith = await scorers["faithfulness"].ascore(
            user_input=question, response=response, retrieved_contexts=contexts
        )
        relevance = await scorers["answer_relevancy"].ascore(
            user_input=question, response=response
        )
        scores.append({
            "question": question,
            "context_precision": cp.value,
            "context_recall": cr.value,
            "faithfulness": faith.value,
            "answer_relevancy": relevance.value,
        })
    eval_df = pd.DataFrame(scores)
    display(eval_df)
    print("\nСередні значення (лише для цієї маленької навчальної вибірки):")
    display(eval_df.drop(columns=["question"]).mean().to_frame("mean_score"))
else:
    print("RAGAS вимкнено. Для платних оцінок увімкніть RUN_LIVE_API=True та RUN_RAGAS_EVAL=True у конфігурації.")

## Інтерпретація результатів та обмеження

| Метрика | Що вимірює | Чого **не** гарантує |
| --- | --- | --- |
| **Context Precision** | Ранжування релевантних фрагментів | Повного покриття всієї необхідної інформації |
| **Context Recall** | Покриття фактичних тверджень еталона контекстом | Правильного формулювання відповіді генератором |
| **Faithfulness** | Підтвердження тверджень контекстом | Істинності фактів у самих джерелах |
| **Answer Relevancy** | Відповідність відповіді запиту | Фактичної правильності відповіді |

**Як QA AI Engineer, звертай увагу:**

1. Retriever може знайти потрібний документ, а генератор усе одно вигадати відповідь.
2. Faithfulness може бути високим, якщо документ сам по собі помилковий.
3. Платний LLM-as-a-Judge не є повністю детермінованим та може бути упередженим.
4. Три позитивні приклади — це **smoke test**, а не доказ якості. Додай згодом десятки прикладів, негативні випадки, контексти з шумом і суперечливими документами.
5. Для регресії зберігай версію моделі, промпту, retrieval settings, датасету, метрики й час запуску.

## Підсумок уроку

- **LangGraph:** `State`, `Node`, `Edge`, `Command`, `interrupt`, checkpoints.
- **Email-агент:** класифікація → обов'язкові кроки → чернетка → human review → симуляція надсилання.
- **RAG:** невеликий корпус → chunks → embeddings → FAISS → response.
- **Evaluation:** golden references та окремі scores `ContextPrecision`, `ContextRecall`, `Faithfulness`, `AnswerRelevancy`.

**Корисні посилання:**

- [Thinking in LangGraph](https://docs.langchain.com/oss/python/langgraph/thinking-in-langgraph)
- [LangGraph interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts)
- [LangChain retrieval](https://docs.langchain.com/oss/python/deepagents/retrieval)
- [RAGAS metrics](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/)

**Важливо:** API та версії бібліотек змінюються. Перевіряйте їхню документацію перед навчальним запуском після значних оновлень. Жоден OpenAI API-виклик не виконувався під час підготовки цього файла.